#### Topic 5: Joins, Subqueries & CTEs
  - INNER, LEFT, RIGHT, FULL OUTER, CROSS, SEMI, ANTI joins
  - Join on multiple columns
  - Self join
  - Subqueries: scalar, correlated, IN/EXISTS
  - CTE (WITH clause)
  - Set operations: UNION, UNION ALL, INTERSECT, EXCEPT

In [0]:
"""
QUICK REFERENCE:
──────────────────────────────────────────────────────────────────────
JOIN TYPE        | Left unmatched | Right unmatched | Result columns
──────────────────────────────────────────────────────────────────────
INNER            | Excluded       | Excluded        | Both tables
LEFT (OUTER)     | Included+NULL  | Excluded        | Both tables
RIGHT (OUTER)    | Excluded       | Included+NULL   | Both tables
FULL OUTER       | Included+NULL  | Included+NULL   | Both tables
CROSS            | N/A            | N/A             | Both (cartesian)
LEFT SEMI        | Excluded       | N/A             | LEFT only
LEFT ANTI        | Included       | N/A             | LEFT only
──────────────────────────────────────────────────────────────────────

NOT IN with NULLs: if subquery returns a NULL, entire NOT IN → FALSE/UNKNOWN
→ Safer to use NOT EXISTS or filter NULLs: WHERE x NOT IN (SELECT y ... WHERE y IS NOT NULL)

UNION vs UNION ALL:
  UNION     → deduplicates (uses Sort + Aggregate internally → slower)
  UNION ALL → no dedup (faster, preferred when duplicates are OK)

CTE (WITH clause):
  - Improves readability; same as inline view in terms of execution
  - Can reference earlier CTEs within the same WITH block
  - NOT materialized by default in Spark (recomputed if referenced multiple times)
──────────────────────────────────────────────────────────────────────
"""


In [0]:
# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW employees AS
    SELECT * FROM VALUES
        (1, 'Alice',  10, 95000),
        (2, 'Bob',    20, 72000),
        (3, 'Carol',  10, 110000),
        (4, 'Dave',   30, 60000),
        (5, 'Eve',    20, 85000),
        (6, 'Frank',  10, 98000),
        (7, 'Grace',  NULL, 63000)   -- no dept
    AS t(emp_id, name, dept_id, salary)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW departments AS
    SELECT * FROM VALUES
        (10, 'Engineering', 'New York'),
        (20, 'Marketing',   'Chicago'),
        (30, 'HR',          'Dallas'),
        (40, 'Finance',     'Boston')    -- no employees
    AS t(dept_id, dept_name, location)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW projects AS
    SELECT * FROM VALUES
        (101, 'Cloud Migration',  1),
        (102, 'SEO Campaign',     2),
        (103, 'Data Platform',    1),
        (104, 'Brand Refresh',    5),
        (105, 'HR Automation',    4)
    AS t(project_id, project_name, lead_emp_id)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW managers AS
    SELECT * FROM VALUES
        (1, 'Alice', 3),     -- Alice managed by Carol
        (2, 'Bob',   5),     -- Bob managed by Eve
        (3, 'Carol', NULL),  -- Carol is top-level
        (4, 'Dave',  NULL),
        (5, 'Eve',   NULL)
    AS t(emp_id, name, manager_id)
""")


##### 1. INNER JOIN  — only rows that match in BOTH tables

In [0]:
# ---------------------------------------------------------------------------
# 1. INNER JOIN  — only rows that match in BOTH tables
# ---------------------------------------------------------------------------
print("=== 1. INNER JOIN ===")
spark.sql("""
    SELECT e.emp_id, e.name, e.salary, d.dept_name, d.location
    FROM employees e
    INNER JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY e.emp_id
""").show()
# Grace (NULL dept_id) and Finance dept (no employees) are excluded


##### 2. LEFT JOIN  — all rows from LEFT table + matches from RIGHT

In [0]:
# ---------------------------------------------------------------------------
# 2. LEFT JOIN  — all rows from LEFT table + matches from RIGHT
#    Left-unmatched rows get NULLs for right columns
# ---------------------------------------------------------------------------
print("=== 2. LEFT JOIN (all employees, even without dept) ===")
spark.sql("""
    SELECT e.emp_id, e.name, e.salary, d.dept_name, d.location
    FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY e.emp_id
""").show()
# Grace appears with NULL dept_name, NULL location


##### 3. RIGHT JOIN  — all rows from RIGHT + matches from LEFT

In [0]:
# ---------------------------------------------------------------------------
# 3. RIGHT JOIN  — all rows from RIGHT + matches from LEFT
# ---------------------------------------------------------------------------
print("=== 3. RIGHT JOIN (all depts, even without employees) ===")
spark.sql("""
    SELECT d.dept_name, d.location, e.name, e.salary
    FROM employees e
    RIGHT JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY d.dept_name
""").show()
# Finance appears with NULL employee info


##### 4. FULL OUTER JOIN  — all rows from BOTH, NULLs where no match

In [0]:
# ---------------------------------------------------------------------------
# 4. FULL OUTER JOIN  — all rows from BOTH, NULLs where no match
# ---------------------------------------------------------------------------
print("=== 4. FULL OUTER JOIN ===")
spark.sql("""
    SELECT
        COALESCE(e.emp_id, -1)            AS emp_id,
        COALESCE(e.name, 'No Employee')   AS emp_name,
        COALESCE(d.dept_name, 'No Dept')  AS dept_name
    FROM employees e
    FULL OUTER JOIN departments d ON e.dept_id = d.dept_id
    ORDER BY emp_id
""").show()


##### 5. CROSS JOIN  — cartesian product (every row x every row)

In [0]:
# ---------------------------------------------------------------------------
# 5. CROSS JOIN  — cartesian product (every row x every row)
#    No ON clause. Use carefully — m * n rows!
# ---------------------------------------------------------------------------
print("=== 5. CROSS JOIN (cartesian product) ===")
spark.sql("""
    SELECT e.name AS employee, d.dept_name AS department
    FROM (SELECT name FROM employees WHERE emp_id <= 2) e
    CROSS JOIN (SELECT dept_name FROM departments) d
    ORDER BY employee, department
""").show()


##### 6. LEFT SEMI JOIN  — like INNER but returns ONLY LEFT columns

In [0]:
# ---------------------------------------------------------------------------
# 6. LEFT SEMI JOIN  — like INNER but returns ONLY LEFT columns
#    Equivalent to: WHERE EXISTS (subquery)
#    Useful for existence checks without duplicating left rows
# ---------------------------------------------------------------------------
print("=== 6. LEFT SEMI JOIN (employees who lead projects) ===")
spark.sql("""
    SELECT e.emp_id, e.name, e.salary
    FROM employees e
    LEFT SEMI JOIN projects p ON e.emp_id = p.lead_emp_id
    ORDER BY e.emp_id
""").show()


##### 7. LEFT ANTI JOIN  — opposite of SEMI; rows from LEFT with NO match in RIGHT

In [0]:
# ---------------------------------------------------------------------------
# 7. LEFT ANTI JOIN  — opposite of SEMI; rows from LEFT with NO match in RIGHT
#    Equivalent to: WHERE NOT EXISTS (subquery)
# ---------------------------------------------------------------------------
print("=== 7. LEFT ANTI JOIN (employees NOT leading any project) ===")
spark.sql("""
    SELECT e.emp_id, e.name, e.salary
    FROM employees e
    LEFT ANTI JOIN projects p ON e.emp_id = p.lead_emp_id
    ORDER BY e.emp_id
""").show()


##### 8. SELF JOIN  — table joined with itself

In [0]:
# ---------------------------------------------------------------------------
# 8. SELF JOIN  — table joined with itself
#    Classic use case: employee → manager hierarchy
# ---------------------------------------------------------------------------
print("=== 8. SELF JOIN (employee with their manager name) ===")
spark.sql("""
    SELECT
        e.emp_id,
        e.name           AS employee,
        m.name           AS manager,
        m.emp_id         AS manager_id
    FROM managers e
    LEFT JOIN managers m ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
""").show()


##### 9. JOIN ON MULTIPLE COLUMNS

In [0]:
# ---------------------------------------------------------------------------
# 9. JOIN ON MULTIPLE COLUMNS
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW sales AS
    SELECT * FROM VALUES
        ('Q1', 10, 150000),
        ('Q1', 20,  90000),
        ('Q2', 10, 180000),
        ('Q2', 20,  95000)
    AS t(quarter, dept_id, revenue)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW targets AS
    SELECT * FROM VALUES
        ('Q1', 10, 140000),
        ('Q1', 20, 100000),
        ('Q2', 10, 170000)
    AS t(quarter, dept_id, target)
""")

print("=== 9. Multi-column JOIN ===")
spark.sql("""
    SELECT s.quarter, s.dept_id, s.revenue, t.target,
           s.revenue - t.target AS vs_target
    FROM sales s
    LEFT JOIN targets t ON s.quarter = t.quarter AND s.dept_id = t.dept_id
    ORDER BY s.quarter, s.dept_id
""").show()

##### 10. SUBQUERIES

In [0]:
# ---------------------------------------------------------------------------
# 10. SUBQUERIES
# ---------------------------------------------------------------------------

# 10a. Scalar subquery — returns a single value
print("=== 10a. Scalar Subquery ===")
spark.sql("""
    SELECT name, salary,
           (SELECT AVG(salary) FROM employees) AS company_avg,
           salary - (SELECT AVG(salary) FROM employees) AS diff_from_avg
    FROM employees
    ORDER BY diff_from_avg DESC
""").show()


# 10b. IN subquery — filter using a list from another query
print("=== 10b. IN Subquery ===")
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE emp_id IN (
        SELECT lead_emp_id FROM projects
    )
    ORDER BY salary DESC
""").show()

# 10c. NOT IN  — be careful: if subquery returns ANY NULL, entire result is empty
print("=== 10c. NOT IN Subquery (watch for NULLs!) ===")
spark.sql("""
    SELECT name, salary
    FROM employees
    WHERE emp_id NOT IN (
        SELECT lead_emp_id FROM projects WHERE lead_emp_id IS NOT NULL
    )
    ORDER BY emp_id
""").show()

# 10d. EXISTS / NOT EXISTS  — safer alternative to IN/NOT IN (ignores NULLs)
print("=== 10d. EXISTS Subquery ===")
spark.sql("""
    SELECT e.name, e.salary
    FROM employees e
    WHERE EXISTS (
        SELECT 1 FROM projects p WHERE p.lead_emp_id = e.emp_id
    )
    ORDER BY e.salary DESC
""").show()

# 10e. Inline view (derived table) — subquery in FROM clause
print("=== 10e. Derived Table / Inline View ===")
spark.sql("""
    SELECT dept_summary.dept_name,
           dept_summary.avg_sal,
           dept_summary.headcount
    FROM (
        SELECT d.dept_name,
               ROUND(AVG(e.salary), 0) AS avg_sal,
               COUNT(e.emp_id)         AS headcount
        FROM employees e
        JOIN departments d ON e.dept_id = d.dept_id
        GROUP BY d.dept_name
    ) dept_summary
    WHERE dept_summary.headcount >= 2
    ORDER BY avg_sal DESC
""").show()


##### 11. CTE — Common Table Expressions  (WITH clause)

In [0]:
# ---------------------------------------------------------------------------
# 11. CTE — Common Table Expressions  (WITH clause)
#    Cleaner, reusable, readable alternative to nested subqueries
#    Multiple CTEs separated by commas
# ---------------------------------------------------------------------------
print("=== 11. CTE (WITH clause) ===")
spark.sql("""
    WITH dept_stats AS (
        SELECT
            d.dept_id,
            d.dept_name,
            COUNT(e.emp_id)         AS headcount,
            ROUND(AVG(e.salary), 0) AS avg_salary,
            MAX(e.salary)           AS max_salary,
            MIN(e.salary)           AS min_salary
        FROM departments d
        LEFT JOIN employees e ON d.dept_id = e.dept_id
        GROUP BY d.dept_id, d.dept_name
    ),
    above_avg_depts AS (
        SELECT dept_id, dept_name, avg_salary
        FROM dept_stats
        WHERE avg_salary > (SELECT AVG(salary) FROM employees)
    ),
    ranked_employees AS (
        SELECT
            e.name,
            e.salary,
            d.dept_name,
            ds.avg_salary AS dept_avg
        FROM employees e
        JOIN departments d ON e.dept_id = d.dept_id
        JOIN dept_stats  ds ON d.dept_id = ds.dept_id
    )
    SELECT
        re.dept_name,
        re.name,
        re.salary,
        re.dept_avg,
        ROUND(re.salary - re.dept_avg, 0) AS diff_from_dept_avg
    FROM ranked_employees re
    JOIN above_avg_depts aad ON re.dept_name = aad.dept_name
    ORDER BY re.dept_name, re.salary DESC
""").show()

##### 12. SET OPERATIONS

In [0]:
# ---------------------------------------------------------------------------
# 12. SET OPERATIONS
#    UNION ALL     → combine all rows (includes duplicates)
#    UNION         → combine + deduplicate (slower, triggers shuffle)
#    INTERSECT     → rows in BOTH result sets
#    INTERSECT ALL → with duplicates (less common)
#    EXCEPT        → rows in first but NOT in second
#    EXCEPT ALL    → with duplicates
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW set_a AS
    SELECT * FROM VALUES (1,'Alpha'), (2,'Beta'), (3,'Gamma'), (2,'Beta')
    AS t(id, label)
""")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW set_b AS
    SELECT * FROM VALUES (2,'Beta'), (3,'Gamma'), (4,'Delta')
    AS t(id, label)
""")

print("=== 12a. UNION ALL (all rows, keeps duplicates) ===")
spark.sql("SELECT * FROM set_a UNION ALL SELECT * FROM set_b").show()

print("=== 12b. UNION (deduplicated) ===")
spark.sql("SELECT * FROM set_a UNION SELECT * FROM set_b").show()

print("=== 12c. INTERSECT (in both) ===")
spark.sql("SELECT * FROM set_a INTERSECT SELECT * FROM set_b").show()

print("=== 12d. EXCEPT (in A but not B) ===")
spark.sql("SELECT * FROM set_a EXCEPT SELECT * FROM set_b").show()


In [0]:
spark.stop()